In [1]:
# ==========================================================
# COSC2669 / COSC2816
# Individual Task 1 - Fraud Detection Experiment
# Student number: s4160984
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
# ==========================================================
# 1. LOAD DATASETS
# ==========================================================

credit_df = pd.read_csv("creditcard.csv")
vehicle_df = pd.read_csv("fraud_oracle.csv")

print("Credit Card Dataset Shape:", credit_df.shape)
print("Vehicle Fraud Dataset Shape:", vehicle_df.shape)


Credit Card Dataset Shape: (284807, 31)
Vehicle Fraud Dataset Shape: (15420, 33)


In [3]:
# ==========================================================
# 2. INITIAL DATA EXPLORATION
# ==========================================================

print("\n==============================")
print("CREDIT CARD DATASET")
print("==============================")

print(credit_df.head())

print("\nData types:")
print(credit_df.dtypes)

print("\nMissing values:")
print(credit_df.isnull().sum())

print("\nTotal missing values:")
print(credit_df.isnull().sum().sum())

print("\nDuplicate rows:")
print(credit_df.duplicated().sum())

print("\nClass distribution:")
print(credit_df["Class"].value_counts())

print("\nClass percentage:")
print(
    credit_df["Class"]
    .value_counts(normalize=True)
    .mul(100)
)

print("\n==============================")
print("VEHICLE FRAUD DATASET")
print("==============================")

print(vehicle_df.head())

print("\nData types:")
print(vehicle_df.dtypes)

print("\nMissing values:")
print(vehicle_df.isnull().sum())

print("\nTotal missing values:")
print(vehicle_df.isnull().sum().sum())

print("\nDuplicate rows:")
print(vehicle_df.duplicated().sum())

print("\nClass distribution:")
print(vehicle_df["FraudFound_P"].value_counts())

print("\nClass percentage:")
print(
    vehicle_df["FraudFound_P"]
    .value_counts(normalize=True)
    .mul(100)
)



CREDIT CARD DATASET
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26

In [4]:
# ==========================================================
# 3. REMOVE DUPLICATES FROM CREDIT CARD DATASET
# ==========================================================

credit_clean = credit_df.drop_duplicates().copy()

print("\nOriginal Credit Card Rows:", len(credit_df))

print(
    "Duplicate Rows Removed:",
    len(credit_df) - len(credit_clean)
)

print(
    "Rows After Duplicate Removal:",
    len(credit_clean)
)

print("\nClass Distribution After Removing Duplicates:")
print(credit_clean["Class"].value_counts())

print("\nClass Percentage After Removing Duplicates:")
print(
    credit_clean["Class"]
    .value_counts(normalize=True)
    .mul(100)
)


Original Credit Card Rows: 284807
Duplicate Rows Removed: 1081
Rows After Duplicate Removal: 283726

Class Distribution After Removing Duplicates:
0    283253
1       473
Name: Class, dtype: int64

Class Percentage After Removing Duplicates:
0    99.83329
1     0.16671
Name: Class, dtype: float64


In [5]:
# ==========================================================
# 4. PREPARE CREDIT CARD DATA
# ==========================================================

X_credit = credit_clean.drop(columns=["Class"])
y_credit = credit_clean["Class"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_credit,
    y_credit,
    test_size=0.20,
    random_state=42,
    stratify=y_credit
)

print("\nCredit Card Training Shape:", Xc_train.shape)
print("Credit Card Testing Shape:", Xc_test.shape)

print("\nTraining Class Distribution:")
print(yc_train.value_counts())

print("\nTesting Class Distribution:")
print(yc_test.value_counts())


Credit Card Training Shape: (226980, 30)
Credit Card Testing Shape: (56746, 30)

Training Class Distribution:
0    226602
1       378
Name: Class, dtype: int64

Testing Class Distribution:
0    56651
1       95
Name: Class, dtype: int64


In [6]:
# ==========================================================
# 5. PREPARE VEHICLE FRAUD DATA
# ==========================================================

vehicle_clean = vehicle_df.copy()

X_vehicle = vehicle_clean.drop(
    columns=["FraudFound_P"]
)

y_vehicle = vehicle_clean["FraudFound_P"]

vehicle_numeric_columns = (
    X_vehicle
    .select_dtypes(include=["int64", "float64"])
    .columns
    .tolist()
)

vehicle_categorical_columns = (
    X_vehicle
    .select_dtypes(include=["object"])
    .columns
    .tolist()
)

print("\nVehicle Numeric Columns:")
print(vehicle_numeric_columns)

print("\nVehicle Categorical Columns:")
print(vehicle_categorical_columns)

Xv_train, Xv_test, yv_train, yv_test = train_test_split(
    X_vehicle,
    y_vehicle,
    test_size=0.20,
    random_state=42,
    stratify=y_vehicle
)

print("\nVehicle Training Shape:", Xv_train.shape)
print("Vehicle Testing Shape:", Xv_test.shape)


Vehicle Numeric Columns:
['WeekOfMonth', 'WeekOfMonthClaimed', 'Age', 'PolicyNumber', 'RepNumber', 'Deductible', 'DriverRating', 'Year']

Vehicle Categorical Columns:
['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'BasePolicy']

Vehicle Training Shape: (12336, 32)
Vehicle Testing Shape: (3084, 32)


In [7]:
# ==========================================================
# 6. VEHICLE DATA PREPROCESSING
# ==========================================================

vehicle_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            vehicle_numeric_columns
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            vehicle_categorical_columns
        )
    ]
) 

In [8]:
# ==========================================================
# 7. MODEL EVALUATION FUNCTION
# ==========================================================

def evaluate_model(
    model_name,
    model,
    X_test,
    y_test
):

    predictions = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )
    
    print("\n==================================")
    print(model_name)
    print("==================================")

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_test,
            predictions
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

    auc = np.nan

    if hasattr(model, "predict_proba"):

        probabilities = model.predict_proba(
            X_test
        )[:, 1]

        auc = roc_auc_score(
            y_test,
            probabilities
        )

        print(
            "ROC-AUC  :",
            round(auc, 4)
        )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": auc
    }

In [9]:
# ==========================================================
# 8. DECISION TREE - CREDIT CARD DATA
# ==========================================================

credit_tree = DecisionTreeClassifier(
    random_state=42
)

credit_tree.fit(
    Xc_train,
    yc_train
)

credit_tree_results = evaluate_model(
    "Credit Card - Decision Tree",
    credit_tree,
    Xc_test,
    yc_test
)


Credit Card - Decision Tree

Confusion Matrix:
[[56625    26]
 [   28    67]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.72      0.71      0.71        95

    accuracy                           1.00     56746
   macro avg       0.86      0.85      0.86     56746
weighted avg       1.00      1.00      1.00     56746

Accuracy : 0.999
Precision: 0.7204
Recall   : 0.7053
F1 Score : 0.7128
ROC-AUC  : 0.8524


In [10]:
# ==========================================================
# 9. SCALE CREDIT CARD DATA FOR NEURAL NETWORK
# ==========================================================

credit_scaler = StandardScaler()

Xc_train_scaled = credit_scaler.fit_transform(
    Xc_train
)

Xc_test_scaled = credit_scaler.transform(
    Xc_test
)

In [11]:
# ==========================================================
# 10. NEURAL NETWORK - CREDIT CARD DATA
# ==========================================================

credit_mlp = MLPClassifier(
    random_state=42,
    max_iter=300
)

credit_mlp.fit(
    Xc_train_scaled,
    yc_train
)

credit_mlp_results = evaluate_model(
    "Credit Card - Neural Network",
    credit_mlp,
    Xc_test_scaled,
    yc_test
)


Credit Card - Neural Network

Confusion Matrix:
[[56647     4]
 [   27    68]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.94      0.72      0.81        95

    accuracy                           1.00     56746
   macro avg       0.97      0.86      0.91     56746
weighted avg       1.00      1.00      1.00     56746

Accuracy : 0.9995
Precision: 0.9444
Recall   : 0.7158
F1 Score : 0.8144
ROC-AUC  : 0.9479


In [12]:
# ==========================================================
# 11. DECISION TREE - VEHICLE FRAUD DATA
# ==========================================================

vehicle_tree = Pipeline(
    steps=[
        (
            "preprocessor",
            vehicle_preprocessor
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

vehicle_tree.fit(
    Xv_train,
    yv_train
)

vehicle_tree_results = evaluate_model(
    "Vehicle Fraud - Decision Tree",
    vehicle_tree,
    Xv_test,
    yv_test
)


Vehicle Fraud - Decision Tree

Confusion Matrix:
[[2758  141]
 [ 131   54]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      2899
           1       0.28      0.29      0.28       185

    accuracy                           0.91      3084
   macro avg       0.62      0.62      0.62      3084
weighted avg       0.91      0.91      0.91      3084

Accuracy : 0.9118
Precision: 0.2769
Recall   : 0.2919
F1 Score : 0.2842
ROC-AUC  : 0.6216


In [13]:
# ==========================================================
# 12. NEURAL NETWORK - VEHICLE FRAUD DATA
# ==========================================================

vehicle_mlp = Pipeline(
    steps=[
        (
            "preprocessor",
            vehicle_preprocessor
        ),
        (
            "classifier",
            MLPClassifier(
                random_state=42,
                max_iter=300
            )
        )
    ]
)

vehicle_mlp.fit(
    Xv_train,
    yv_train
)

vehicle_mlp_results = evaluate_model(
    "Vehicle Fraud - Neural Network",
    vehicle_mlp,
    Xv_test,
    yv_test
)



Vehicle Fraud - Neural Network

Confusion Matrix:
[[2807   92]
 [ 152   33]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      2899
           1       0.26      0.18      0.21       185

    accuracy                           0.92      3084
   macro avg       0.61      0.57      0.59      3084
weighted avg       0.91      0.92      0.91      3084

Accuracy : 0.9209
Precision: 0.264
Recall   : 0.1784
F1 Score : 0.2129
ROC-AUC  : 0.7948


In [14]:
# ==========================================================
# 13. FINAL MODEL COMPARISON
# ==========================================================

results = pd.DataFrame([
    credit_tree_results,
    credit_mlp_results,
    vehicle_tree_results,
    vehicle_mlp_results
])

print("\n==================================")
print("FINAL MODEL COMPARISON")
print("==================================")

print(
    results.round(4)
)

results.to_csv(
    "model_results.csv",
    index=False
)


FINAL MODEL COMPARISON
                            Model  Accuracy  Precision  Recall      F1  \
0     Credit Card - Decision Tree    0.9990     0.7204  0.7053  0.7128   
1    Credit Card - Neural Network    0.9995     0.9444  0.7158  0.8144   
2   Vehicle Fraud - Decision Tree    0.9118     0.2769  0.2919  0.2842   
3  Vehicle Fraud - Neural Network    0.9209     0.2640  0.1784  0.2129   

   ROC_AUC  
0   0.8524  
1   0.9479  
2   0.6216  
3   0.7948  
